In [ ]:
import pandas as pd
import numpy as np 
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression


1. train-test split (month=6)

In [27]:
def split(df,month):
    df["month-year"] = pd.to_datetime(df["YearMonth"])
    # Sort chronologically
    df = df.sort_values("month-year")
    latest_month = df["month-year"].max()
    print("Test month:", latest_month)
    # Create test set: latest month only
    test_df = df[
        df["month-year"] == latest_month
    ]

    # Create training set: previous 6 months
    train_start = latest_month - pd.DateOffset(months=month)
    train_df = df[
        (df["month-year"] < latest_month) &
        (df["month-year"] >= train_start)
    ]
    print("Training period:")
    print(train_df["month-year"].min(), "to", train_df["month-year"].max())

    print("Testing period:")
    print(test_df["month-year"].min(), "to", test_df["month-year"].max())
    return train_df, test_df

In [14]:
df = pd.read_csv("data/df_cleaned.csv")
train_df, test_df = split(df, month=6)

Test month: 2026-05-01 00:00:00
Training period:
2025-11-01 00:00:00 to 2026-04-01 00:00:00
Testing period:
2026-05-01 00:00:00 to 2026-05-01 00:00:00


2. Loading 

In [ ]:
# target ='ClosePrice'

# X_train = train_df.drop(columns=[target, "month-year","YearMonth"])
# y_train = train_df[target]

# X_test = test_df.drop(columns=[target, "month-year","YearMonth"])
# y_test = test_df[target]


In [ ]:
# numeric_cols = X_train.select_dtypes(
#     include=["int64", "float64"]
# ).columns.tolist()


# categorical_cols = X_train.select_dtypes(
#     include=["object", "category"]
# ).columns.tolist()

# print("All:", len(X_train.columns))
# print("Numerical:", len(numeric_cols))
# print("Categorical:", len(categorical_cols))

All: 56
Numerical: 22
Categorical: 34


In [20]:
def prepare_data(train_df, test_df):
    target ='ClosePrice'
    X_train = train_df.drop(columns=[target, "month-year","YearMonth"])
    y_train = train_df[target]

    X_test = test_df.drop(columns=[target, "month-year","YearMonth"])
    y_test = test_df[target]

    numeric_cols = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()


    categorical_cols = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    return X_train, y_train, X_test, y_test, numeric_cols, categorical_cols    

In [ ]:
# preprocessor = ColumnTransformer(
#     transformers=[
#         (
#             "num",
#             StandardScaler(),
#             numeric_cols
#         ),
#         (
#             "cat",
#             OneHotEncoder(
#                 handle_unknown="ignore"
#             ),
#             categorical_cols
#         )
#     ]
# )

In [ ]:
# baseline_model = Pipeline(
#     steps=[
#         (
#             "preprocessing",
#             preprocessor
#         ),
#         (
#             "model",
#             LinearRegression()
#         )
#     ]
# )
# baseline_model.fit(
#     X_train,
#     y_train
# )

# y_pred = baseline_model.predict(
#     X_test
# )


# from sklearn.metrics import r2_score

# r2 = r2_score(
#     y_test,
#     y_pred
# )


# print("Baseline Linear Regression R²:", r2)

Baseline Linear Regression R²: -0.6880779410436269


In [15]:
def train(X_train, y_train, X_test, y_test):
    preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        )
        ]
    )
    
    baseline_model = Pipeline(
        steps=[
            (
                "preprocessing",
                preprocessor
            ),
            (
                "model",
                LinearRegression()
            )
        ]
    )
    baseline_model.fit(
        X_train,
        y_train
    )

    y_pred = baseline_model.predict(
        X_test
    )
    r2 = r2_score(
        y_test,
        y_pred
    )
    print("Linear Regression R²:", r2)

In [23]:
def main(df,month=6):
    print(month)
    train_df, test_df = split(df, month) 
    X_train, y_train, X_test, y_test, numeric_cols, categorical_cols = prepare_data(train_df, test_df)
    train(X_train, y_train, X_test, y_test)

In [29]:
main(df, month=3)

3
Test month: 2026-05-01 00:00:00
Training period:
2026-02-01 00:00:00 to 2026-04-01 00:00:00
Testing period:
2026-05-01 00:00:00 to 2026-05-01 00:00:00
Linear Regression R²: -0.3726465071363909


In [30]:
main(df, month=1)

1
Test month: 2026-05-01 00:00:00
Training period:
2026-04-01 00:00:00 to 2026-04-01 00:00:00
Testing period:
2026-05-01 00:00:00 to 2026-05-01 00:00:00
Linear Regression R²: -0.2811791136896067


month = 6 \
Test month: 2026-05-01\
Training period: 2025-11-01 to 2026-04-01 \
Baseline Linear Regression R²: -0.6880779410436269